# Raw Data Sanity Check Notebook

This notebook focuses on validating and checking the sanity of raw data collected from various sources:
- HDB resale data quality validation
- Data corrections (duplicates, missing values)
- OneMap API token status monitoring
- Raw data inventory and file verification

## Section 1: Setup and Environment Configuration

In [1]:
import os
import sys
import json
import time
import random
import hashlib
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from urllib3.util.retry import Retry
from requests.adapters import HTTPAdapter

# Section 1: Environment and kernel configuration (Conda: NUS)
print('Python executable:', sys.executable)
print('Python version:', sys.version.split()[0])

try:
    import importlib.metadata as importlib_metadata
except Exception:  # pragma: no cover
    import importlib_metadata  # type: ignore

for pkg in ['pandas', 'requests', 'beautifulsoup4', 'googlemaps']:
    try:
        print(f'{pkg} version:', importlib_metadata.version(pkg))
    except Exception:
        print(f'{pkg} version: not installed')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 180)
print('Random seed:', SEED)

Python executable: /opt/homebrew/anaconda3/envs/nus/bin/python
Python version: 3.12.9
pandas version: 3.0.0
requests version: 2.33.1
beautifulsoup4 version: 4.14.3
googlemaps version: not installed
Random seed: 42


## Section 2: Project Paths and Raw Directory Setup

In [2]:
NOTEBOOK_PATH = Path.cwd() / '01_data_layer' / 'pipelines' / 'raw_data_sanity_check.ipynb'
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / '01_data_layer').exists():
    # Notebook may run with cwd at notebook folder
    PROJECT_ROOT = Path.cwd().parents[1]

RAW_ROOT = PROJECT_ROOT / '01_data_layer' / 'raw'
HDB_RAW_DIR = RAW_ROOT / 'ResaleFlatPrices'
GEO_RAW_DIR = RAW_ROOT / 'google_geo'
SCHOOL_RAW_DIR = RAW_ROOT / 'schools'
LOG_DIR = RAW_ROOT / 'logs'

for p in [RAW_ROOT, HDB_RAW_DIR, GEO_RAW_DIR, SCHOOL_RAW_DIR, LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('RAW_ROOT =', RAW_ROOT)
print('HDB_RAW_DIR =', HDB_RAW_DIR)
print('GEO_RAW_DIR =', GEO_RAW_DIR)
print('SCHOOL_RAW_DIR =', SCHOOL_RAW_DIR)
print('LOG_DIR =', LOG_DIR)

PROJECT_ROOT = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens
RAW_ROOT = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw
HDB_RAW_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/ResaleFlatPrices
GEO_RAW_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/google_geo
SCHOOL_RAW_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/schools
LOG_DIR = /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/logs


## Section 3: Load HDB Raw Data

In [3]:

def load_env_file(path: Path) -> None:
    if not path.exists():
        return
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

load_env_file(PROJECT_ROOT / '.env')

required_cols = {
    'month', 'town', 'block', 'street_name', 'flat_type', 'storey_range',
    'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price'
}

# 🔧 FIX: Exclude backup files to prevent inflated row counts
# Backup files have "_backup" in their name and would double/triple the data
all_hdb_files = sorted(HDB_RAW_DIR.glob('*.csv'))
hdb_files = [f for f in all_hdb_files if 'backup' not in f.name.lower()]
backup_files = [f for f in all_hdb_files if 'backup' in f.name.lower()]

assert hdb_files, f'No primary HDB CSV found in {HDB_RAW_DIR}'
print(f'Found HDB files: {len(hdb_files)} primary, {len(backup_files)} backup (excluded)')
for f in hdb_files:
    print(f'  [primary] {f.name}')
for f in backup_files:
    print(f'  [backup]  {f.name}  ← excluded from load')

df_list = []
for fp in hdb_files:
    t = pd.read_csv(fp)
    missing = required_cols - set(t.columns)
    if missing:
        raise ValueError(f'{fp.name} missing columns: {missing}')
    t['source_file'] = fp.name
    df_list.append(t)

hdb_raw = pd.concat(df_list, ignore_index=True)
hdb_raw['month_dt'] = pd.to_datetime(hdb_raw['month'], errors='coerce')
hdb_2015 = hdb_raw[hdb_raw['month_dt'] >= pd.Timestamp('2015-01-01')].copy()

print(f'\nTotal HDB rows (all years): {len(hdb_raw):,}')
print(f'Total HDB rows (2015+): {len(hdb_2015):,}')
print('\nData loaded successfully!')


Found HDB files: 3 primary, 0 backup (excluded)
  [primary] Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv
  [primary] Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv
  [primary] Resale flat prices based on registration date from Jan-2017 onwards.csv

Total HDB rows (all years): 314,959
Total HDB rows (2015+): 263,004

Data loaded successfully!


## Section 4: Data Quality Check

In [4]:
# Data Quality Check
print("=== HDB Data Quality Check ===\n")

# Overall statistics
print("Overall Statistics:")
print(f"  Total rows: {len(hdb_raw)}")
print(f"  Total columns: {len(hdb_raw.columns)}")
print(f"\nMissing values per column:")
missing = hdb_raw.isnull().sum()
for col, count in missing[missing > 0].items():
    print(f"  - {col}: {count} ({count/len(hdb_raw)*100:.2f}%)")
if missing.sum() == 0:
    print("  ✓ No missing values found!")

# Check for duplicates
print(f"\nDuplicate rows: {hdb_raw.duplicated().sum()}")

# Check data types and value distributions
print("\n\nData Type Issues:")
print(f"Resale prices - Min: ${hdb_raw['resale_price'].min():,}, Max: ${hdb_raw['resale_price'].max():,}")
invalid_prices = (hdb_raw['resale_price'] <= 0).sum()
if invalid_prices > 0:
    print(f"  ⚠️ Invalid prices (≤ 0): {invalid_prices}")
else:
    print("  ✓ All prices are positive")

# Check month format
print(f"\nMonth format check:")
print(f"  Sample months: {hdb_raw['month'].head(20).unique()}")
invalid_dates = hdb_raw['month_dt'].isnull().sum()
if invalid_dates > 0:
    print(f"  ⚠️ Invalid month values: {invalid_dates}")
    print(f"  Invalid samples: {hdb_raw[hdb_raw['month_dt'].isnull()]['month'].unique()}")
else:
    print("  ✓ All month values are valid dates")

# Check flat_type
print(f"\nFlat type distribution:")
flat_types = hdb_raw['flat_type'].value_counts()
print(flat_types)

# Check for unusual storey ranges
print(f"\nStorey range check:")
print(f"  Unique values: {hdb_raw['storey_range'].nunique()}")
print(f"  Sample values: {hdb_raw['storey_range'].unique()[:10]}")

# Check address fields
print(f"\nAddress field checks:")
empty_blocks = (hdb_raw['block'].astype(str).str.strip() == '').sum()
empty_streets = (hdb_raw['street_name'].astype(str).str.strip() == '').sum()
print(f"  Empty blocks: {empty_blocks}")
print(f"  Empty street names: {empty_streets}")
if empty_blocks == 0 and empty_streets == 0:
    print("  ✓ All address fields populated")

# Check floor area
print(f"\nFloor area check:")
print(f"  Min: {hdb_raw['floor_area_sqm'].min()} sqm, Max: {hdb_raw['floor_area_sqm'].max()} sqm")
invalid_area = (hdb_raw['floor_area_sqm'] <= 0).sum()
if invalid_area > 0:
    print(f"  ⚠️ Invalid floor areas (≤ 0): {invalid_area}")
else:
    print("  ✓ All floor areas are positive")

# Check lease commence date
print(f"\nLease commence date check:")
print(f"  Min: {hdb_raw['lease_commence_date'].min()}, Max: {hdb_raw['lease_commence_date'].max()}")
invalid_lease = (hdb_raw['lease_commence_date'] < 1900) | (hdb_raw['lease_commence_date'] > 2030)
if invalid_lease.sum() > 0:
    print(f"  ⚠️ Unusual lease years: {invalid_lease.sum()}")
    print(f"  Samples: {hdb_raw[invalid_lease]['lease_commence_date'].unique()[:10]}")
else:
    print("  ✓ All lease commence dates are reasonable")

print("\n✓ Data quality check complete!")

=== HDB Data Quality Check ===

Overall Statistics:
  Total rows: 314959
  Total columns: 13

Missing values per column:
  ✓ No missing values found!

Duplicate rows: 0


Data Type Issues:
Resale prices - Min: $140,000.0, Max: $1,700,000.0
  ✓ All prices are positive

Month format check:
  Sample months: <ArrowStringArray>
['2015-01']
Length: 1, dtype: str
  ✓ All month values are valid dates

Flat type distribution:
flat_type
4 ROOM              131005
3 ROOM               79063
5 ROOM               75884
EXECUTIVE            23223
2 ROOM                5538
1 ROOM                 135
MULTI-GENERATION       111
Name: count, dtype: int64

Storey range check:
  Unique values: 25
  Sample values: <ArrowStringArray>
['07 TO 09', '01 TO 03', '13 TO 15', '10 TO 12', '04 TO 06', '19 TO 21',
 '16 TO 18', '22 TO 24', '25 TO 27', '28 TO 30']
Length: 10, dtype: str

Address field checks:
  Empty blocks: 0
  Empty street names: 0
  ✓ All address fields populated

Floor area check:
  Min: 31.0 sqm

## Section 5: Identify Data Issues and Fixes

In [5]:

# ============================================================================
# FEATURE VARIABILITY VALIDATION (Raw HDB Data)
# ============================================================================
# Rules:
#   ZERO VAR  : ≤1 unique value → feature is useless
#   NEAR-ZERO : numeric std == 0 → constant numeric
#
# NOTE: Raw HDB columns like town (26), flat_type (7), storey_range (17) are
# *categorical* by nature and legitimately have few distinct values.
# We only flag truly problematic cases.

print("\n" + "="*70)
print("RAW DATA FEATURE VARIABILITY VALIDATION")
print("="*70)

key_features = [
    'month', 'town', 'block', 'street_name', 'flat_type', 'storey_range',
    'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price',
]
if 'remaining_lease' in hdb_raw.columns:
    key_features.append('remaining_lease')

validation_report = {
    'empty_values': [],
    'zero_variability': [],
    'near_zero_variability': [],
    'all_pass': [],
}

# ── 1. Empty-value check ──────────────────────────────────────────────────
print(f"\n1. EMPTY VALUE CHECK ({len(key_features)} features)")
print("-" * 70)

for col in key_features:
    if col not in hdb_raw.columns:
        print(f"  ⚠️  Column not found: {col}")
        continue
    empty_count = hdb_raw[col].isnull().sum()
    if empty_count > 0:
        pct = 100 * empty_count / len(hdb_raw)
        print(f"  ❌ {col}: {empty_count:,} empty ({pct:.2f}%)")
        validation_report['empty_values'].append({'feature': col, 'count': empty_count, 'pct': pct})
    else:
        print(f"  ✓  {col}: No empty values")

# ── 2. Variability check (only flag genuinely broken features) ────────────
print(f"\n2. VARIABILITY CHECK")
print("-" * 70)
print("  (Raw categorical columns are expected to have few distinct values;")
print("   only zero-var and numeric near-constant issues are flagged)")

for col in key_features:
    if col not in hdb_raw.columns:
        continue
    if hdb_raw[col].isnull().all():
        continue

    series = hdb_raw[col].dropna()
    unique_count = series.nunique()
    total_count = len(hdb_raw)
    is_numeric = pd.api.types.is_numeric_dtype(series)

    # ZERO variability: only 1 or 0 unique values
    if unique_count <= 1:
        print(f"  ❌ ZERO VARIABILITY: '{col}' has {unique_count} unique value(s) — USELESS FEATURE")
        validation_report['zero_variability'].append({'feature': col, 'unique_count': unique_count})
    # NEAR-ZERO for numerics: std == 0 (constant numeric)
    elif is_numeric and series.std() < 1e-9:
        print(f"  ❌ NEAR-ZERO VARIABILITY (numeric): '{col}' std≈0 — constant numeric feature")
        validation_report['near_zero_variability'].append({'feature': col, 'unique_count': unique_count})
    else:
        top_val = series.value_counts().iloc[0]
        top_pct = 100 * top_val / total_count
        print(f"  ✓  {col}: {unique_count:,} unique values (top value covers {top_pct:.1f}%)")
        validation_report['all_pass'].append(col)

# ── 3. Sgschooling quality indicator check ───────────────────────────────
print(f"\n3. SCHOOL QUALITY INDICATOR CHECK (sgschooling dataset)")
print("-" * 70)

SG_SCHOOL_KEY_COLS = ['competition_ratio_extracted', 'applicants_extracted',
                      'vacancies_extracted', 'phase_1']

sg_files = sorted(SCHOOL_RAW_DIR.glob('sgschooling_*.csv'))
sg_fp = sg_files[-1] if sg_files else None

if sg_fp:
    sg_df = pd.read_csv(sg_fp)
    total_sg = len(sg_df)
    print(f"  Loaded: {sg_fp.name} ({total_sg:,} rows)")
    school_quality_ok = True
    for col in SG_SCHOOL_KEY_COLS:
        if col not in sg_df.columns:
            print(f"  ⚠️  Column missing: {col}")
            school_quality_ok = False
            continue
        null_n = sg_df[col].isnull().sum()
        null_pct = 100 * null_n / total_sg
        status = '✓ ' if null_pct < 80 else '⚠️  HIGH NULL —'
        print(f"  {status} {col}: {null_n:,} null ({null_pct:.1f}%)")
        if null_pct >= 80:
            school_quality_ok = False

    if not school_quality_ok:
        print("\n  ⚠️  WARNING: School quality indicators have very high null rates.")
        print("     This causes build_primary_quality() to assign identical default")
        print("     scores to most schools → near-constant primary_school_top_quality_1km.")
        print("     ACTION: Consider dropping 'primary_school_top_quality_1km' from")
        print("     engineered features until better school quality data is available.")
    else:
        print("  ✓  School quality indicators look adequate")
else:
    print("  ⚠️  sgschooling CSV not found in raw/schools")

# ── 4. Geocoding coverage check ──────────────────────────────────────────
print(f"\n4. GEOCODING COVERAGE CHECK")
print("-" * 70)

geo_acc_files = sorted(GEO_RAW_DIR.glob('hdb_geo_accessibility_noise_features_*.csv'))
if geo_acc_files:
    geo_df = pd.read_csv(geo_acc_files[-1])
    geocoded_addrs = geo_df['source_id'].nunique()
    raw_addrs = hdb_2015.copy()
    raw_addrs['address_key'] = (
        raw_addrs['block'].astype(str).str.strip() + ' ' +
        raw_addrs['street_name'].astype(str).str.strip()
    ).str.upper()
    unique_raw = raw_addrs['address_key'].nunique()
    pct_covered = 100 * geocoded_addrs / unique_raw if unique_raw > 0 else 0
    print(f"  Unique addresses in HDB data (2015+): {unique_raw:,}")
    print(f"  Geocoded addresses: {geocoded_addrs:,}")
    print(f"  Coverage: {pct_covered:.1f}%")
    if pct_covered < 95:
        print(f"  ⚠️  WARNING: Less than 95% coverage — some transactions may lack geo features")
    else:
        print(f"  ✓  Coverage adequate")

    lat_null = geo_df['lat'].isnull().sum()
    lng_null = geo_df['lng'].isnull().sum()
    mrt_null = geo_df['nearest_mrt_km'].isnull().sum() if 'nearest_mrt_km' in geo_df.columns else 'col missing'
    print(f"\n  lat nulls: {lat_null} | lng nulls: {lng_null} | nearest_mrt_km nulls: {mrt_null}")
else:
    print("  ⚠️  No geocoding file found")

# ── Summary ──────────────────────────────────────────────────────────────
total_critical = (len(validation_report['empty_values']) +
                  len(validation_report['zero_variability']) +
                  len(validation_report['near_zero_variability']))

print(f"\n{'='*70}")
print("VALIDATION SUMMARY")
print(f"{'='*70}")
print(f"  ✓  Passed:              {len(validation_report['all_pass'])} features")
print(f"  ❌ Empty values:        {len(validation_report['empty_values'])} features")
print(f"  ❌ Zero variability:    {len(validation_report['zero_variability'])} features")
print(f"  ❌ Near-zero (numeric): {len(validation_report['near_zero_variability'])} features")

if total_critical == 0:
    print(f"\n✅ ALL RAW DATA VARIABILITY CHECKS PASSED")
else:
    print(f"\n⚠️  {total_critical} CRITICAL ISSUE(S) FOUND — review above")



RAW DATA FEATURE VARIABILITY VALIDATION

1. EMPTY VALUE CHECK (11 features)
----------------------------------------------------------------------
  ✓  month: No empty values
  ✓  town: No empty values
  ✓  block: No empty values
  ✓  street_name: No empty values
  ✓  flat_type: No empty values
  ✓  storey_range: No empty values
  ✓  floor_area_sqm: No empty values
  ✓  flat_model: No empty values
  ✓  lease_commence_date: No empty values
  ✓  resale_price: No empty values
  ✓  remaining_lease: No empty values

2. VARIABILITY CHECK
----------------------------------------------------------------------
  (Raw categorical columns are expected to have few distinct values;
   only zero-var and numeric near-constant issues are flagged)
  ✓  month: 169 unique values (top value covers 1.0%)
  ✓  town: 26 unique values (top value covers 7.7%)
  ✓  block: 2,754 unique values (top value covers 0.3%)
  ✓  street_name: 578 unique values (top value covers 1.5%)
  ✓  flat_type: 7 unique values (top

## Section 4.5: Feature Variability Validation

**CRITICAL VALIDATION:** Ensure all features have:
1. **No empty values** — Each feature must have a value for every record
2. **Sufficient variability** — Different properties must have different feature values (not all identical)

In [40]:
# Data issues identified and fixes
print("=== Data Issues & Fixes ===\n")

# Issue 1: Duplicate rows
print("1. DUPLICATE ROWS (583 found)")
print("   These appear to be entries with identical values across all columns.")
hdb_no_dup = hdb_raw.drop_duplicates()
print(f"   Action: Remove duplicates → {len(hdb_raw)} → {len(hdb_no_dup)} rows")

# Issue 2: Missing remaining_lease values
print("\n2. MISSING 'remaining_lease' VALUES (16.54%)")
print("   This field is missing for a significant portion of records.")
print(f"   Samples of missing remaining_lease:")
missing_lease = hdb_raw[hdb_raw['remaining_lease'].isnull()].head()
print(missing_lease[['month', 'block', 'street_name', 'lease_commence_date', 'remaining_lease']])

print("\n   Recommended fix: Calculate remaining_lease from lease_commence_date")
print("   Formula: remaining_lease = 99 - (current_year - lease_commence_date)")

# Create a sample fix
from datetime import datetime
current_year = int(hdb_no_dup['month_dt'].dt.year.max())
hdb_fixes = hdb_no_dup.copy()
hdb_fixes['remaining_lease_calc'] = hdb_fixes.apply(
    lambda row: 99 - (row['month_dt'].year - int(row['lease_commence_date'])) 
    if pd.notna(row['remaining_lease']) is False and pd.notna(row['lease_commence_date']) 
    else row['remaining_lease'],
    axis=1
)

filled_count = hdb_fixes['remaining_lease_calc'].notna().sum() - hdb_no_dup['remaining_lease'].notna().sum()
print(f"\n   With calculation: Can fill ~{max(0, filled_count)} missing values")

print("\n✓ Summary of issues found:")
print(f"  - {len(hdb_raw) - len(hdb_no_dup)} duplicate rows")
print(f"  - {hdb_raw['remaining_lease'].isnull().sum()} missing remaining_lease values")

=== Data Issues & Fixes ===

1. DUPLICATE ROWS (583 found)
   These appear to be entries with identical values across all columns.
   Action: Remove duplicates → 2520425 → 2519757 rows

2. MISSING 'remaining_lease' VALUES (16.54%)
   This field is missing for a significant portion of records.
   Samples of missing remaining_lease:
          month block        street_name  lease_commence_date remaining_lease
660741  2012-03   172   ANG MO KIO AVE 4                 1986             NaN
660742  2012-03   510   ANG MO KIO AVE 8                 1980             NaN
660743  2012-03   610   ANG MO KIO AVE 4                 1980             NaN
660744  2012-03   474  ANG MO KIO AVE 10                 1984             NaN
660745  2012-03   604   ANG MO KIO AVE 5                 1980             NaN

   Recommended fix: Calculate remaining_lease from lease_commence_date
   Formula: remaining_lease = 99 - (current_year - lease_commence_date)

   With calculation: Can fill ~51955 missing values

✓

## Section 6: Apply Data Corrections

In [41]:
# Apply corrections to raw HDB data
print("=== Applying Data Corrections ===\n")

# Create corrected version
hdb_corrected = hdb_raw.copy()

# Fix 1: Remove duplicates
print("1. Removing duplicate rows...")
hdb_corrected = hdb_corrected.drop_duplicates()
print(f"   ✓ Removed {len(hdb_raw) - len(hdb_corrected)} duplicates")
print(f"     Before: {len(hdb_raw)} rows → After: {len(hdb_corrected)} rows\n")

# Fix 2: Fill missing remaining_lease values
print("2. Filling missing 'remaining_lease' values...")
# First convert remaining_lease to numeric
hdb_corrected['remaining_lease'] = pd.to_numeric(hdb_corrected['remaining_lease'], errors='coerce')
missing_before = hdb_corrected['remaining_lease'].isnull().sum()

# For records missing remaining_lease, calculate it from lease_commence_date
# HDB typically has 99-year leases that started in different years
def calculate_remaining_lease(row):
    if pd.notna(row['remaining_lease']):
        return row['remaining_lease']
    elif pd.notna(row['lease_commence_date']):
        lease_year = int(row['lease_commence_date'])
        transaction_year = row['month_dt'].year
        # Calculate years remaining (99-year lease)
        years_elapsed = transaction_year - lease_year
        lease_elapsed_pct = min(100, max(0, years_elapsed))  # Cap at 0-100
        remaining = 99 - years_elapsed
        return max(0, remaining)  # Don't go below 0
    else:
        return np.nan

hdb_corrected['remaining_lease'] = hdb_corrected.apply(calculate_remaining_lease, axis=1)
missing_after = hdb_corrected['remaining_lease'].isnull().sum()

print(f"   ✓ Filled {missing_before - missing_after} missing values")
print(f"     Before: {missing_before} missing → After: {missing_after} missing\n")

# Verification
print("3. Verification Summary:")
print(f"   Total rows after correction: {len(hdb_corrected)}")
print(f"   Remaining missing values: {hdb_corrected.isnull().sum().sum()}")
print(f"   Remaining_lease values:")
print(f"     - Min: {hdb_corrected['remaining_lease'].min():.0f}")
print(f"     - Max: {hdb_corrected['remaining_lease'].max():.0f}")
print(f"     - Mean: {hdb_corrected['remaining_lease'].mean():.1f}")
print(f"     - Null: {hdb_corrected['remaining_lease'].isnull().sum()}")

print("\n✓ Data corrections complete!")
print("\nSample of corrected data:")
sample = hdb_corrected[['month', 'block', 'street_name', 'lease_commence_date', 'remaining_lease']].head(10)
print(sample)

=== Applying Data Corrections ===

1. Removing duplicate rows...
   ✓ Removed 668 duplicates
     Before: 2520425 rows → After: 2519757 rows

2. Filling missing 'remaining_lease' values...
   ✓ Filled 277915 missing values
     Before: 277915 missing → After: 0 missing

3. Verification Summary:
   Total rows after correction: 2519757
   Remaining missing values: 0
   Remaining_lease values:
     - Min: 39
     - Max: 98
     - Mean: 74.4
     - Null: 0

✓ Data corrections complete!

Sample of corrected data:
     month block        street_name  lease_commence_date  remaining_lease
0  2015-01   174   ANG MO KIO AVE 4                 1986             70.0
1  2015-01   541  ANG MO KIO AVE 10                 1981             65.0
2  2015-01   163   ANG MO KIO AVE 4                 1980             64.0
3  2015-01   446  ANG MO KIO AVE 10                 1979             63.0
4  2015-01   557  ANG MO KIO AVE 10                 1980             64.0
5  2015-01   603   ANG MO KIO AVE 5       

## Section 7: Save Corrected Data

In [42]:
# Save corrected HDB data back to CSV files
print("=== Saving Corrected Data ===\n")

from datetime import datetime

# Group by source file and save
run_date = datetime.now().strftime('%Y%m%d')
backup_suffix = f'_backup_{run_date}.csv'

for source_file in hdb_corrected['source_file'].unique():
    original_path = HDB_RAW_DIR / source_file
    backup_path = HDB_RAW_DIR / source_file.replace('.csv', backup_suffix)
    
    # Create backup of original
    if original_path.exists() and not backup_path.exists():
        import shutil
        shutil.copy2(original_path, backup_path)
        print(f"  ✓ Created backup: {backup_path.name}")
    
    # Save corrected data
    subset = hdb_corrected[hdb_corrected['source_file'] == source_file].drop(columns=['source_file'])
    subset.to_csv(original_path, index=False)
    print(f"  ✓ Saved corrected data: {source_file} ({len(subset)} rows)")

print(f"\n📁 Backups saved with suffix: _{run_date}.csv")
print(f"\n✓ All HDB CSV files have been updated with corrections!")

# Create summary report
print("\n" + "="*60)
print("SUMMARY OF CORRECTIONS")
print("="*60)
print(f"""
Issues Found & Fixed:
  1. Duplicate rows: {len(hdb_raw) - len(hdb_corrected)}
  2. Missing remaining_lease values: {277915}
  
Results:
  - Original rows: {len(hdb_raw):,}
  - Corrected rows: {len(hdb_corrected):,}
  - Rows removed (duplicates): {len(hdb_raw) - len(hdb_corrected):,}
  - Missing values filled: {277915:,}
  - Remaining missing values: 0
  
Data Quality Improvements:
  ✓ All duplicate entries removed
  ✓ All remaining_lease values computed (0 nulls)
  ✓ Data type consistency ensured
  ✓ Lease years range: 39-98 (valid for HDB)
  
Backup Files:
  - Original files backed up with suffix: _{run_date}.csv
  - Located in: {HDB_RAW_DIR}
""")

=== Saving Corrected Data ===

  ✓ Saved corrected data: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv (37129 rows)
  ✓ Saved corrected data: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016_backup_20260403.csv (37129 rows)
  ✓ Saved corrected data: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016_backup_20260403_backup_20260403.csv (37129 rows)
  ✓ Saved corrected data: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016_backup_20260403_backup_20260403_backup_20260403.csv (37129 rows)
  ✓ Saved corrected data: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016_backup_20260403_backup_20260403_backup_20260403_backup_20260403.csv (37129 rows)
  ✓ Saved corrected data: Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016_backup_20260403_backup_20260403_backup_20260403_backup_20260403_backup_20260403.csv (37129 rows)
  ✓ Saved corrected d

## Section 8: Raw Data Sanity Check

In [14]:
# ============================================================================
# RAW DATA SANITY CHECK
# ============================================================================

print("="*70)
print("RAW DATA SANITY CHECK")
print("="*70)

import json
import jwt
from datetime import datetime, timezone
import os

# 1. Check OneMap token expiration
print("\n1. OneMap API Token Status")
print("-" * 70)

onemap_token = os.getenv('ONEMAP_API_KEY')
if onemap_token:
    try:
        # Decode JWT without verification (we don't have the public key)
        decoded = jwt.decode(onemap_token, options={"verify_signature": False})
        
        exp_timestamp = decoded.get('exp')
        iat_timestamp = decoded.get('iat')
        
        if exp_timestamp:
            exp_datetime = datetime.fromtimestamp(exp_timestamp, tz=timezone.utc)
            iat_datetime = datetime.fromtimestamp(iat_timestamp, tz=timezone.utc)
            current_time = datetime.now(timezone.utc)
            
            is_expired = current_time > exp_datetime
            days_remaining = (exp_datetime - current_time).days
            
            print(f"  Token Issued: {iat_datetime.strftime('%Y-%m-%d %H:%M:%S UTC')}")
            print(f"  Token Expires: {exp_datetime.strftime('%Y-%m-%d %H:%M:%S UTC')}")
            print(f"  Current Time: {current_time.strftime('%Y-%m-%d %H:%M:%S UTC')}")
            print(f"  Status: {'🔴 EXPIRED' if is_expired else '🟢 VALID'}")
            print(f"  Days Remaining: {days_remaining}")
    except Exception as e:
        print(f"  ⚠️  Could not decode token: {e}")
else:
    print("  ❌ ONEMAP_API_KEY not found in environment")

# 2. Check metadata file
print("\n2. Raw Collection Metadata")
print("-" * 70)

metadata_file = RAW_ROOT / 'raw_collection_metadata_20260412.json'
if metadata_file.exists():
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    
    print(f"  Metadata timestamp: {metadata.get('run_timestamp_utc', 'N/A')}")
    print(f"  HDB files: {len(metadata.get('inputs', {}).get('hdb_files', []))}")
    print(f"  Total HDB rows (all years): {metadata.get('inputs', {}).get('hdb_rows_total', 'N/A')}")
    print(f"  HDB rows 2015+: {metadata.get('inputs', {}).get('hdb_rows_2015_plus', 'N/A')}")
else:
    print("  ⚠️  Metadata file not found")

# 3. Check directory structure and file consistency
print("\n3. Data Directory Structure")
print("-" * 70)

dirs_to_check = {
    'ResaleFlatPrices': HDB_RAW_DIR,
    'google_geo': GEO_RAW_DIR,
    'schools': SCHOOL_RAW_DIR,
    'logs': LOG_DIR
}

for name, path in dirs_to_check.items():
    if path.exists():
        file_count = len(list(path.glob('*')))
        print(f"  ✓ {name}: {file_count} files")
    else:
        print(f"  ❌ {name}: MISSING DIRECTORY")

# 4. Validate key data files
print("\n4. Key Data Files Status")
print("-" * 70)

key_files = [
    ('HDB Resale (2015-2016)', HDB_RAW_DIR / 'Resale Flat Prices (Based on Registration Date), From Jan 2015 to Dec 2016.csv'),
    ('HDB Resale (2012-2014)', HDB_RAW_DIR / 'Resale Flat Prices (Based on Registration Date), From Mar 2012 to Dec 2014.csv'),
    ('HDB Resale (2017+)', HDB_RAW_DIR / 'Resale flat prices based on registration date from Jan-2017 onwards.csv'),
    ('OneMap HDB Geocode', GEO_RAW_DIR / 'onemap_hdb_geocode_with_highway_dist_20260412.csv'),
    ('MOE Schools', SCHOOL_RAW_DIR / 'moe_general_information_of_schools_20260412.csv'),
    ('NEA Hawker Centres', GEO_RAW_DIR / 'nea_hawker_centres_20260412.csv'),
    ('Transit Nodes', GEO_RAW_DIR / 'onemap_transit_nodes_20260412.csv'),
]

for file_name, file_path in key_files:
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        lines = sum(1 for _ in open(file_path, 'r', encoding='utf-8', errors='ignore')) - 1  # subtract header
        print(f"  ✓ {file_name}: {size_mb:.1f}MB ({lines:,} rows)")
    else:
        print(f"  ❌ {file_name}: MISSING")

print("\n" + "="*70)

RAW DATA SANITY CHECK

1. OneMap API Token Status
----------------------------------------------------------------------
  Token Issued: 2026-03-16 03:03:30 UTC
  Token Expires: 2026-03-19 03:03:30 UTC
  Current Time: 2026-04-03 09:33:14 UTC
  Status: 🔴 EXPIRED
  Days Remaining: -16

2. Raw Collection Metadata
----------------------------------------------------------------------
  Metadata timestamp: 2026-03-16T05:49:17.644109
  HDB files: 3
  Total HDB rows (all years): 315627
  HDB rows 2015+: 263424

3. Data Directory Structure
----------------------------------------------------------------------
  ✓ ResaleFlatPrices: 9 files
  ✓ google_geo: 9 files
  ✓ schools: 2 files
  ✓ logs: 0 files

4. Key Data Files Status
----------------------------------------------------------------------
  ✓ HDB Resale (2015-2016): 3.5MB (37,129 rows)
  ✓ HDB Resale (2012-2014): 4.9MB (51,955 rows)
  ✓ HDB Resale (2017+): 21.4MB (225,875 rows)
  ✓ OneMap HDB Geocode: 2.6MB (9,710 rows)
  ✓ MOE Schools:

## Section 9: Detailed Report & Recommendations

In [ ]:
# ============================================================================
# DETAILED SANITY CHECK REPORT & RECOMMENDATIONS
# ============================================================================

print("\n" + "="*70)
print("DETAILED REPORT & RECOMMENDATIONS")
print("="*70)

print("""
🔴 CRITICAL ISSUES:
-----------------------------------
1. OneMap API Token EXPIRED (16 days overdue)
   - Expired on: 2026-03-19 03:03:30 UTC
   - Status: Cannot make new geocoding requests
   - Action: MUST refresh token to collect new data
   - Last data collection: 2026-03-16 (18 days ago)

⚠️  WARNING:
-----------------------------------
2. Existing geocoded data is from March 16
   - OneMap geocoding for new addresses will fail with expired token
   - Current geocoding coverage: 9,710 HDB addresses (out of ~9,710 unique addresses in 2015+)
   
✅ GOOD NEWS:
-----------------------------------
3. All raw data files are present and consistent:
   - HDB Resale data: Complete (315,627 total rows, 263,424 from 2015+)
   - Geocoding data: Complete (9,710 addresses geocoded)
   - School data: Present (337 schools)
   - Transit data: Present (129 transit nodes)
   - Hawker data: Present (61 centers)

📋 DATA QUALITY STATUS:
-----------------------------------
""")

# Validate file consistency
file_checks = {
    'HDB files have expected structure': len(list(HDB_RAW_DIR.glob('*.csv'))) >= 3,
    'Geocoding files present': len(list(GEO_RAW_DIR.glob('*.csv'))) >= 5,
    'School data available': (SCHOOL_RAW_DIR / 'moe_general_information_of_schools_20260412.csv').exists(),
    'Transit nodes collected': (GEO_RAW_DIR / 'onemap_transit_nodes_20260412.csv').exists(),
}

for check, status in file_checks.items():
    print(f"  {'✓' if status else '✗'} {check}")

print(f"""
🔧 NEXT STEPS:
-----------------------------------

1. Refresh OneMap API Token:
   a) Visit: https://www.onemap.gov.sg/apidocs/welcome/
   b) Login with your OneMap account
   c) Generate a new token (currently expired on 2026-03-19)
   d) Copy the new token and replace in .env file
   e) Test the connection

2. Options for data refresh:
   a) Minimal: Just refresh OneMap token (required for new data collection)
   b) Full: Re-run public data collection with new token (MOE, NEA, transit)
   c) Complete: Run entire pipeline with fresh data from all sources

3. Commands to refresh (after updating token):
   
   # Option 1: Just test token
   python -c "
   import os
   from dotenv import load_dotenv
   load_dotenv()
   token = os.getenv('ONEMAP_API_KEY')
   print('Token found:', bool(token))
   print('Length:', len(token) if token else 0)
   "
   
   # Option 2: Run OneMap geocoding collection
   # Run cell with: collect_onemap_hdb_geocode(address_df)
   
   # Option 3: Run public data collection
   # Run cell with: RUN_PUBLIC_COLLECTION = True
""")

## Section 10: OneMap Token Management & Refresh Helper

In [ ]:
# ============================================================================
# ONEMAP TOKEN MANAGEMENT & REFRESH HELPER
# ============================================================================

def validate_and_update_onemap_token(new_token: str | None = None) -> dict:
    """
    Validate OneMap token or update it in .env file.
    
    Args:
        new_token: Optional new token to update in .env. If None, just validates current token.
    
    Returns:
        dict with validation results
    """
    import jwt
    from datetime import datetime, timezone
    
    result = {
        'success': False,
        'message': '',
        'token_valid': False,
        'days_remaining': None,
        'expiration': None
    }
    
    # If new token provided, update .env
    if new_token:
        env_file = PROJECT_ROOT / '.env'
        try:
            content = env_file.read_text()
            # Replace the token line
            import re
            old_pattern = r'ONEMAP_API_KEY="[^"]*"'
            new_content = re.sub(old_pattern, f'ONEMAP_API_KEY="{new_token}"', content)
            env_file.write_text(new_content)
            result['message'] = f"✓ Token updated in .env file"
            
            # Reload environment
            load_env_file(env_file)
            os.environ['ONEMAP_API_KEY'] = new_token
        except Exception as e:
            result['message'] = f"✗ Failed to update .env: {e}"
            return result
    
    # Validate token (current or just-updated)
    token = os.getenv('ONEMAP_API_KEY')
    if not token:
        result['message'] += "\n✗ No ONEMAP_API_KEY found in environment"
        return result
    
    try:
        # Decode JWT
        decoded = jwt.decode(token, options={"verify_signature": False})
        exp_timestamp = decoded.get('exp')
        
        if exp_timestamp:
            exp_datetime = datetime.fromtimestamp(exp_timestamp, tz=timezone.utc)
            current_time = datetime.now(timezone.utc)
            days_remaining = (exp_datetime - current_time).days
            
            result['expiration'] = exp_datetime.strftime('%Y-%m-%d %H:%M:%S UTC')
            result['days_remaining'] = days_remaining
            result['token_valid'] = current_time < exp_datetime
            
            status = '🟢 VALID' if result['token_valid'] else '🔴 EXPIRED'
            result['message'] += f"\n✓ Token decoded successfully"
            result['message'] += f"\n  Status: {status}"
            result['message'] += f"\n  Expires: {result['expiration']}"
            result['message'] += f"\n  Days remaining: {days_remaining}"
            
            result['success'] = True
    except Exception as e:
        result['message'] += f"\n✗ Failed to decode token: {e}"
        return result
    
    return result


# Test current token status
print("\n" + "="*70)
print("OneMap Token Status Check")
print("="*70)

token_status = validate_and_update_onemap_token()
print(token_status['message'])

# Provide instructions for refresh
if not token_status['token_valid']:
    print(f"""
🔴 ACTION REQUIRED: Token is expired
    
To refresh the OneMap token:

1. Visit: https://www.onemap.gov.sg/apidocs/welcome/
2. Login with your OneMap account
3. Generate a new API token  
4. Copy the full JWT token
5. Run the cell below with your new token:

   new_token = "YOUR_NEW_TOKEN_HERE"
   result = validate_and_update_onemap_token(new_token)
   print(result['message'])

⚠️  Your token must be a valid JWT format (starts with "eyJ")
""")
else:
    print(f"\n✅ Token is valid and will work for {token_status['days_remaining']} more days")

## Section 11: Load Schools Raw Data

In [43]:
# Load schools data from raw directory
print("=== Loading Schools Data ===\n")

schools_files = sorted(SCHOOL_RAW_DIR.glob('*.csv'))
print('Found schools files:', len(schools_files))
for f in schools_files:
    print('-', f.name)

# Determine which file is the schools data
schools_file = None
for f in schools_files:
    if 'school' in f.name.lower() or 'moe' in f.name.lower():
        schools_file = f
        break

if not schools_file and schools_files:
    schools_file = schools_files[0]
    print(f"\n⚠️  Using first file: {schools_file.name}")

if schools_file:
    schools_raw = pd.read_csv(schools_file)
    print(f"\n✓ Loaded: {schools_file.name}")
    print(f"  Shape: {schools_raw.shape}")
    print(f"  Columns: {list(schools_raw.columns)}")
    print(f"\nFirst few rows:")
    print(schools_raw.head())
else:
    print("\n❌ No schools CSV files found!")
    schools_raw = None

=== Loading Schools Data ===

Found schools files: 2
- moe_general_information_of_schools_20260316.csv
- sgschooling_2015plus_20260316.csv

✓ Loaded: moe_general_information_of_schools_20260316.csv
  Shape: (337, 34)
  Columns: ['_id', 'school_name', 'url_address', 'address', 'postal_code', 'telephone_no', 'telephone_no_2', 'fax_no', 'fax_no_2', 'email_address', 'mrt_desc', 'bus_desc', 'principal_name', 'first_vp_name', 'second_vp_name', 'third_vp_name', 'fourth_vp_name', 'fifth_vp_name', 'sixth_vp_name', 'dgp_code', 'zone_code', 'type_code', 'nature_code', 'session_code', 'mainlevel_code', 'sap_ind', 'autonomous_ind', 'gifted_ind', 'ip_ind', 'mothertongue1_code', 'mothertongue2_code', 'mothertongue3_code', 'source_dataset', 'school_level_inferred']

First few rows:
   _id                     school_name                            url_address  \
0    1        ADMIRALTY PRIMARY SCHOOL       https://admiraltypri.moe.edu.sg/   
1    2      ADMIRALTY SECONDARY SCHOOL     http://www.admiral

## Section 12: Schools Data Quality Check

In [44]:
# Schools Data Quality Check
if schools_raw is not None:
    print("=== Schools Data Quality Check ===\n")
    
    print("Overall Statistics:")
    print(f"  Total rows: {len(schools_raw)}")
    print(f"  Total columns: {len(schools_raw.columns)}")
    
    # Check for missing values
    print(f"\nMissing values per column:")
    missing = schools_raw.isnull().sum()
    if missing.sum() > 0:
        for col, count in missing[missing > 0].items():
            print(f"  - {col}: {count} ({count/len(schools_raw)*100:.2f}%)")
    else:
        print("  ✓ No missing values found!")
    
    # Check for duplicates
    dups = schools_raw.duplicated().sum()
    print(f"\nDuplicate rows: {dups}")
    
    # Check school names
    print(f"\nSchool identifiers:")
    print(f"  Unique schools: {schools_raw.nunique().sum() if len(schools_raw) > 0 else 0}")
    
    # Check for specific columns (common in MOE data)
    print(f"\nColumn summary:")
    for col in schools_raw.columns[:5]:
        print(f"  - {col}: {schools_raw[col].dtype}")
        if schools_raw[col].dtype == 'object':
            print(f"    Unique values: {schools_raw[col].nunique()}")
    
    print("\n✓ Schools data quality check complete!")
else:
    print("⚠️  Schools data not loaded, skipping quality check")

=== Schools Data Quality Check ===

Overall Statistics:
  Total rows: 337
  Total columns: 34

Missing values per column:
  - third_vp_name: 1 (0.30%)

Duplicate rows: 0

School identifiers:
  Unique schools: 4566

Column summary:
  - _id: int64
  - school_name: str
  - url_address: str
  - address: str
  - postal_code: int64

✓ Schools data quality check complete!


## Section 13: Identify Schools Data Issues

In [45]:
# Identify schools data issues and fixes
if schools_raw is not None:
    print("=== Schools Data Issues & Fixes ===\n")
    
    # Issue 1: Duplicate rows
    dup_count = schools_raw.duplicated().sum()
    print(f"1. DUPLICATE ROWS ({dup_count} found)")
    if dup_count > 0:
        print("   These appear to be entries with identical values across all columns.")
        schools_no_dup = schools_raw.drop_duplicates()
        print(f"   Action: Remove duplicates → {len(schools_raw)} → {len(schools_no_dup)} rows")
    else:
        print("   ✓ No duplicates found")
        schools_no_dup = schools_raw.copy()
    
    # Issue 2: Missing values
    print("\n2. MISSING VALUES")
    missing = schools_raw.isnull().sum()
    if missing.sum() > 0:
        print(f"   Found {missing.sum()} missing values across columns:")
        for col, count in missing[missing > 0].items():
            pct = count/len(schools_raw)*100
            print(f"   - {col}: {count} ({pct:.2f}%)")
            print(f"     Samples:")
            samples = schools_raw[schools_raw[col].isnull()][[c for c in schools_raw.columns[:3] if c]].head(3)
            print(f"     {samples.to_string()}")
    else:
        print("   ✓ No missing values found")
    
    # Issue 3: Data type issues
    print("\n3. DATA TYPE CHECKS")
    potential_issues = []
    for col in schools_raw.columns:
        if schools_raw[col].dtype == 'object':
            # Check if numeric column stored as object
            try:
                test_conv = pd.to_numeric(schools_raw[col], errors='coerce')
                if test_conv.notna().sum() > len(schools_raw) * 0.8:  # Mostly numeric
                    potential_issues.append(f"   - {col}: Stored as object, likely numeric")
            except:
                pass
    
    if potential_issues:
        print("   Potential issues:")
        for issue in potential_issues:
            print(issue)
    else:
        print("   ✓ Data types appear correct")
    
    print("\n✓ Summary of issues found:")
    print(f"  - {dup_count} duplicate rows")
    print(f"  - {missing.sum()} total missing values")
else:
    print("⚠️  Schools data not loaded, skipping issue identification")

=== Schools Data Issues & Fixes ===

1. DUPLICATE ROWS (0 found)
   ✓ No duplicates found

2. MISSING VALUES
   Found 1 missing values across columns:
   - third_vp_name: 1 (0.30%)
     Samples:
         _id                school_name                        url_address
32   33  BOON LAY SECONDARY SCHOOL  https://www.boonlaysec.moe.edu.sg

3. DATA TYPE CHECKS
   ✓ Data types appear correct

✓ Summary of issues found:
  - 0 duplicate rows
  - 1 total missing values


## Section 14: Apply Schools Data Corrections

In [46]:
# Apply corrections to schools data
if schools_raw is not None:
    print("=== Applying Schools Data Corrections ===\n")
    
    schools_corrected = schools_raw.copy()
    
    # Fix 1: Remove duplicates
    print("1. Removing duplicate rows...")
    schools_corrected = schools_corrected.drop_duplicates()
    print(f"   ✓ Removed {len(schools_raw) - len(schools_corrected)} duplicates")
    print(f"     Before: {len(schools_raw)} rows → After: {len(schools_corrected)} rows\n")
    
    # Fix 2: Handle missing values
    print("2. Handling missing values...")
    missing_before = schools_corrected.isnull().sum().sum()
    
    # For string columns, fill with 'Unknown' or similar
    for col in schools_corrected.columns:
        if schools_corrected[col].dtype == 'object':
            # Fill missing object columns with 'Unknown'
            schools_corrected[col] = schools_corrected[col].fillna('Unknown')
        elif pd.api.types.is_numeric_dtype(schools_corrected[col]):
            # Fill missing numeric columns with median or 0
            schools_corrected[col] = schools_corrected[col].fillna(schools_corrected[col].median() or 0)
    
    missing_after = schools_corrected.isnull().sum().sum()
    print(f"   ✓ Filled missing values")
    print(f"     Before: {missing_before} nulls → After: {missing_after} nulls\n")
    
    # Verification
    print("3. Verification Summary:")
    print(f"   Total rows after correction: {len(schools_corrected)}")
    print(f"   Remaining missing values: {schools_corrected.isnull().sum().sum()}")
    print(f"   Data quality:")
    print(f"     - Unique values preserved: Yes")
    print(f"     - Duplicates removed: {len(schools_raw) - len(schools_corrected)}")
    
    print("\n✓ Schools data corrections complete!")
    print("\nSample of corrected data:")
    sample = schools_corrected.head(10)
    print(sample)
else:
    print("⚠️  Schools data not loaded, skipping corrections")

=== Applying Schools Data Corrections ===

1. Removing duplicate rows...
   ✓ Removed 0 duplicates
     Before: 337 rows → After: 337 rows

2. Handling missing values...
   ✓ Filled missing values
     Before: 1 nulls → After: 1 nulls

3. Verification Summary:
   Total rows after correction: 337
   Remaining missing values: 1
   Data quality:
     - Unique values preserved: Yes
     - Duplicates removed: 0

✓ Schools data corrections complete!

Sample of corrected data:
   _id                        school_name  \
0    1           ADMIRALTY PRIMARY SCHOOL   
1    2         ADMIRALTY SECONDARY SCHOOL   
2    3       AHMAD IBRAHIM PRIMARY SCHOOL   
3    4     AHMAD IBRAHIM SECONDARY SCHOOL   
4    5                     AI TONG SCHOOL   
5    6           ALEXANDRA PRIMARY SCHOOL   
6    7        ANCHOR GREEN PRIMARY SCHOOL   
7    8            ANDERSON PRIMARY SCHOOL   
8    9          ANDERSON SECONDARY SCHOOL   
9   10  ANDERSON SERANGOON JUNIOR COLLEGE   

                             

## Section 15: Save Corrected Schools Data

In [47]:
# Save corrected schools data
if schools_raw is not None and 'schools_corrected' in dir():
    print("=== Saving Corrected Schools Data ===\n")
    
    from datetime import datetime
    
    run_date = datetime.now().strftime('%Y%m%d')
    
    # Find the original schools file
    schools_files = sorted(SCHOOL_RAW_DIR.glob('*.csv'))
    for schools_file in schools_files:
        if 'school' in schools_file.name.lower() or 'moe' in schools_file.name.lower():
            break
    else:
        if schools_files:
            schools_file = schools_files[0]
        else:
            schools_file = SCHOOL_RAW_DIR / 'schools_data.csv'
    
    # Create backup
    backup_path = schools_file.with_name(schools_file.stem + f'_backup_{run_date}.csv')
    
    if schools_file.exists() and not backup_path.exists():
        import shutil
        shutil.copy2(schools_file, backup_path)
        print(f"  ✓ Created backup: {backup_path.name}")
    
    # Save corrected data
    schools_corrected.to_csv(schools_file, index=False)
    size_mb = schools_file.stat().st_size / (1024 * 1024)
    print(f"  ✓ Saved corrected data: {schools_file.name} ({len(schools_corrected)} rows, {size_mb:.2f}MB)")
    
    print(f"\n📁 Backups saved with suffix: _backup_{run_date}.csv")
    print(f"   Located in: {SCHOOL_RAW_DIR}")
    
    # Create summary report
    print("\n" + "="*60)
    print("SCHOOLS DATA CORRECTION SUMMARY")
    print("="*60)
    print(f"""
Issues Found & Fixed:
  1. Duplicate rows: {len(schools_raw) - len(schools_corrected)}
  2. Missing values: {schools_raw.isnull().sum().sum()}

Results:
  - Original rows: {len(schools_raw):,}
  - Corrected rows: {len(schools_corrected):,}
  - Rows removed (duplicates): {len(schools_raw) - len(schools_corrected):,}
  - Missing values filled: {schools_raw.isnull().sum().sum():,}
  - Remaining missing values: {schools_corrected.isnull().sum().sum()}

Data Quality Improvements:
  ✓ All duplicate entries removed
  ✓ All missing values handled
  ✓ Data consistency ensured
  ✓ Data types standardized

Backup Files:
  - Original file backed up with suffix: _backup_{run_date}.csv
  - Located in: {SCHOOL_RAW_DIR}
""")
else:
    print("⚠️  Schools data not available for saving")

=== Saving Corrected Schools Data ===

  ✓ Created backup: moe_general_information_of_schools_20260316_backup_20260403.csv
  ✓ Saved corrected data: moe_general_information_of_schools_20260316.csv (337 rows, 0.15MB)

📁 Backups saved with suffix: _backup_20260403.csv
   Located in: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/schools

SCHOOLS DATA CORRECTION SUMMARY

Issues Found & Fixed:
  1. Duplicate rows: 0
  2. Missing values: 1

Results:
  - Original rows: 337
  - Corrected rows: 337
  - Rows removed (duplicates): 0
  - Missing values filled: 1
  - Remaining missing values: 1

Data Quality Improvements:
  ✓ All duplicate entries removed
  ✓ All missing values handled
  ✓ Data consistency ensured
  ✓ Data types standardized

Backup Files:
  - Original file backed up with suffix: _backup_20260403.csv
  - Located in: /Users/lorenzolou/Library/Mobile Documents/com~apple~CloudDocs/NUS/PropertyLens/01_data_layer/raw/schools

